# Change-Point Detection: Know When the Game Changes

**Goal:** Build a system that actively detects regime changes and restarts exploration.

**Key Idea:** Discounted bandits adapt slowly (~50-100 steps). Change-point detection can trigger immediate restart when a regime shift is detected, recovering faster.

**Commodity example:** Oil price collapse in March 2020 was abrupt. Detecting this change in 5-10 days beats gradual adaptation over 50 days.

## Setup: CUSUM Change-Point Detector

CUSUM (Cumulative Sum) detects when a signal's mean shifts. We'll use it to monitor bandit rewards.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

np.random.seed(42)

class CUSUMDetector:
    """
    CUSUM change-point detector.
    Signals when cumulative deviations exceed threshold.
    """
    def __init__(self, mu0=0.5, k=0.1, h=5.0):
        """
        Args:
            mu0: Baseline mean (pre-change)
            k: Slack parameter (tolerance for noise)
            h: Threshold for detection
        """
        self.mu0 = mu0
        self.k = k
        self.h = h
        self.S = 0  # Cumulative sum

    def update(self, x):
        """Update CUSUM statistic and check for change."""
        self.S = max(0, self.S + (x - self.mu0) - self.k)
        return self.S > self.h  # True if change detected

    def reset(self):
        """Reset detector after change detected."""
        self.S = 0

# Test CUSUM on simulated data with regime change
T = 500
data = np.concatenate([
    np.random.normal(0.5, 0.1, 250),  # Pre-change: mean 0.5
    np.random.normal(0.8, 0.1, 250)   # Post-change: mean 0.8
])

detector = CUSUMDetector(mu0=0.5, k=0.05, h=3.0)
cusum_values = []
detections = []

for t, x in enumerate(data):
    change_detected = detector.update(x)
    cusum_values.append(detector.S)
    if change_detected:
        detections.append(t)
        detector.reset()

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Data with change point
axes[0].plot(data, alpha=0.6, label='Observed data')
axes[0].axvline(250, color='red', linestyle='--', label='True change point')
for d in detections:
    axes[0].axvline(d, color='green', linestyle=':', alpha=0.7)
axes[0].set_ylabel('Value')
axes[0].set_title('Data Stream with Regime Change')
axes[0].legend()
axes[0].grid(alpha=0.3)

# CUSUM statistic
axes[1].plot(cusum_values, label='CUSUM statistic', color='blue')
axes[1].axhline(detector.h, color='red', linestyle='--', label=f'Threshold h={detector.h}')
for d in detections:
    axes[1].axvline(d, color='green', linestyle=':', label='Detection' if d == detections[0] else '')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('CUSUM S(t)')
axes[1].set_title('CUSUM Statistic Exceeds Threshold → Change Detected')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"True change at step 250")
print(f"CUSUM detected change at steps: {detections}")
print(f"Detection lag: {detections[0] - 250} steps" if detections else "No detection")

In [ ]:
learning_objectives(["**CUSUM change-point detection** monitors cumulative deviations to detect regime shifts", "**Combining CUSUM with bandits** enables immediate reset and re-exploration when changes detected", "**Threshold tuning matters:**", "**Faster adaptation than discounting:** CUSUM can detect and restart within 10-20 steps, vs 50-100 for exponential forgetting", "**Trade-off:** Computational overhead (monitoring CUSUM stats) vs faster recovery from regime shifts"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Combine CUSUM with Bandit: Detect Changes → Reset

When CUSUM detects a regime change, reset the bandit's beliefs and re-explore.

In [ ]:
class ThompsonSamplingWithCUSUM:
    """
    Thompson Sampling with CUSUM-based change detection.
    Resets beliefs when regime change detected.
    """
    def __init__(self, n_arms, cusum_k=0.1, cusum_h=5.0):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)
        self.beta_params = np.ones(n_arms)
        self.detectors = [CUSUMDetector(mu0=0.5, k=cusum_k, h=cusum_h) for _ in range(n_arms)]
        self.change_points = []  # Record when changes detected

    def select_arm(self):
        samples = [beta.rvs(a, b) for a, b in zip(self.alpha, self.beta_params)]
        return np.argmax(samples)

    def update(self, arm, reward, t):
        # Update beliefs
        self.alpha[arm] += reward
        self.beta_params[arm] += (1 - reward)

        # Update CUSUM detector for selected arm
        change_detected = self.detectors[arm].update(reward)

        if change_detected:
            # Change detected → reset ALL arms and re-explore
            self.alpha = np.ones(self.n_arms)
            self.beta_params = np.ones(self.n_arms)
            for detector in self.detectors:
                detector.reset()
            self.change_points.append(t)
            return True  # Signal change detected

        return False

# Define non-stationary reward function
def get_reward_probabilities(t):
    """3 regimes with abrupt changes."""
    if t < 500:
        return [0.7, 0.4, 0.3]
    elif t < 1000:
        return [0.3, 0.7, 0.4]
    else:
        return [0.2, 0.3, 0.8]

# Run Thompson Sampling with CUSUM
T = 1500
bandit = ThompsonSamplingWithCUSUM(n_arms=3, cusum_k=0.05, cusum_h=4.0)
selections = []
rewards = []

for t in range(T):
    arm = bandit.select_arm()
    selections.append(arm)

    probs = get_reward_probabilities(t)
    reward = np.random.binomial(1, probs[arm])
    rewards.append(reward)

    change_detected = bandit.update(arm, reward, t)

# Visualize selections and detected changes
plt.figure(figsize=(12, 5))
plt.scatter(range(T), selections, alpha=0.3, s=1, label='Selected arm')
plt.axvline(500, color='red', linestyle='--', alpha=0.5, label='True regime change')
plt.axvline(1000, color='red', linestyle='--', alpha=0.5)
for cp in bandit.change_points:
    plt.axvline(cp, color='green', linestyle=':', alpha=0.7, linewidth=2,
                label='CUSUM detection' if cp == bandit.change_points[0] else '')
plt.xlabel('Time Step')
plt.ylabel('Selected Arm')
plt.title('Thompson Sampling + CUSUM: Detects Changes and Resets')
plt.yticks([0, 1, 2])
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"True regime changes at: 500, 1000")
print(f"CUSUM detected changes at: {bandit.change_points}")
print(f"Total reward: {sum(rewards)}")

## Test on Commodity-Like Data

Simulate commodity returns with realistic regime shifts (crisis, recovery, stability).

In [ ]:
def commodity_regime_simulator(t):
    """
    Simulate 3 commodities with regime-dependent returns.
    Regime 1 (0-400): Normal market
    Regime 2 (400-800): Crisis (WTI crashes)
    Regime 3 (800-1200): Recovery (diversification helps)
    """
    if t < 400:  # Normal market
        return {
            'WTI': np.random.normal(0.6, 0.15),
            'NATGAS': np.random.normal(0.5, 0.20),
            'GOLD': np.random.normal(0.45, 0.10)
        }
    elif t < 800:  # Crisis: oil crashes, gold safe haven
        return {
            'WTI': np.random.normal(0.2, 0.25),     # Bad returns, high vol
            'NATGAS': np.random.normal(0.4, 0.20),
            'GOLD': np.random.normal(0.7, 0.10)     # Flight to safety
        }
    else:  # Recovery: natural gas performs
        return {
            'WTI': np.random.normal(0.5, 0.15),
            'NATGAS': np.random.normal(0.75, 0.18),  # Energy demand recovers
            'GOLD': np.random.normal(0.45, 0.10)
        }

# Map commodities to arms
commodities = ['WTI', 'NATGAS', 'GOLD']

# Run bandit with CUSUM on commodity data
T = 1200
bandit = ThompsonSamplingWithCUSUM(n_arms=3, cusum_k=0.08, cusum_h=5.0)
selections = []
rewards = []
commodity_names = []

for t in range(T):
    arm = bandit.select_arm()
    selected_commodity = commodities[arm]
    selections.append(arm)
    commodity_names.append(selected_commodity)

    # Get returns from regime-dependent distribution
    regime_returns = commodity_regime_simulator(t)
    raw_return = regime_returns[selected_commodity]

    # Normalize to [0, 1] for bandit (clip extreme values)
    reward = np.clip(raw_return, 0, 1)
    rewards.append(reward)

    bandit.update(arm, reward, t)

# Visualize portfolio allocation over time
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Arm selections
axes[0].scatter(range(T), selections, alpha=0.4, s=2)
axes[0].axvline(400, color='red', linestyle='--', alpha=0.5, label='True regime change')
axes[0].axvline(800, color='red', linestyle='--', alpha=0.5)
for cp in bandit.change_points:
    axes[0].axvline(cp, color='green', linestyle=':', linewidth=2,
                    label='CUSUM detection' if cp == bandit.change_points[0] else '')
axes[0].set_ylabel('Selected Commodity')
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(commodities)
axes[0].set_title('Commodity Selection: CUSUM Detects Regime Shifts')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Cumulative rewards
axes[1].plot(np.cumsum(rewards), linewidth=2, label='Cumulative return')
axes[1].axvline(400, color='red', linestyle='--', alpha=0.3)
axes[1].axvline(800, color='red', linestyle='--', alpha=0.3)
for cp in bandit.change_points:
    axes[1].axvline(cp, color='green', linestyle=':', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Time Step (Days)')
axes[1].set_ylabel('Cumulative Reward')
axes[1].set_title('Performance: Adapts to Crisis and Recovery')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Regime structure:")
print("  0-399: Normal (WTI best)")
print("  400-799: Crisis (GOLD safe haven)")
print("  800-1199: Recovery (NATGAS strong)")
print(f"\nDetected changes: {bandit.change_points}")
print(f"Final cumulative return: {sum(rewards):.2f}")

## Compare Detection Sensitivity

Tune CUSUM threshold `h`: too sensitive = false alarms, too insensitive = slow adaptation.

In [ ]:
# Test multiple threshold values
threshold_values = [2.0, 4.0, 6.0, 8.0]

results = []

for h in threshold_values:
    bandit = ThompsonSamplingWithCUSUM(n_arms=3, cusum_k=0.08, cusum_h=h)
    total_reward = 0
    num_detections = 0

    for t in range(T):
        arm = bandit.select_arm()
        regime_returns = commodity_regime_simulator(t)
        reward = np.clip(regime_returns[commodities[arm]], 0, 1)
        total_reward += reward

        if bandit.update(arm, reward, t):
            num_detections += 1

    results.append({
        'h': h,
        'total_reward': total_reward,
        'num_detections': num_detections,
        'change_points': bandit.change_points
    })

# Display results
import pandas as pd

df_results = pd.DataFrame(results)
print("\nCUSUM Threshold Sensitivity Analysis:")
print(df_results[['h', 'total_reward', 'num_detections']])
print("\nTrue regime changes: 2 (at steps 400 and 800)")
print("\nObservations:")
print("- Low h (2.0): Many false alarms → over-resets → worse performance")
print("- Medium h (4.0-6.0): Good balance → detects real changes")
print("- High h (8.0): Misses changes → slow adaptation")

# Visualize trade-off
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_results['h'], df_results['total_reward'], marker='o', linewidth=2)
axes[0].set_xlabel('CUSUM Threshold h')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Reward vs Sensitivity')
axes[0].grid(alpha=0.3)

axes[1].plot(df_results['h'], df_results['num_detections'], marker='o', linewidth=2, color='orange')
axes[1].axhline(2, color='red', linestyle='--', label='True number of changes')
axes[1].set_xlabel('CUSUM Threshold h')
axes[1].set_ylabel('Number of Detections')
axes[1].set_title('Detection Frequency vs Sensitivity')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Interactive Widget: Tune Detection Threshold

Experiment with different CUSUM parameters interactively.

In [ ]:
from ipywidgets import interact, FloatSlider

def run_cusum_bandit(threshold_h, slack_k):
    """
    Run bandit with specified CUSUM parameters and visualize.
    """
    T = 1200
    bandit = ThompsonSamplingWithCUSUM(n_arms=3, cusum_k=slack_k, cusum_h=threshold_h)
    selections = []
    rewards = []

    for t in range(T):
        arm = bandit.select_arm()
        selections.append(arm)
        regime_returns = commodity_regime_simulator(t)
        reward = np.clip(regime_returns[commodities[arm]], 0, 1)
        rewards.append(reward)
        bandit.update(arm, reward, t)

    # Plot
    plt.figure(figsize=(12, 4))
    plt.scatter(range(T), selections, alpha=0.3, s=2)
    plt.axvline(400, color='red', linestyle='--', alpha=0.5, label='True regime change')
    plt.axvline(800, color='red', linestyle='--', alpha=0.5)
    for cp in bandit.change_points:
        plt.axvline(cp, color='green', linestyle=':', linewidth=2,
                    label='Detection' if cp == bandit.change_points[0] else '')
    plt.xlabel('Time')
    plt.ylabel('Selected Arm')
    plt.yticks([0, 1, 2], commodities)
    plt.title(f'h={threshold_h:.1f}, k={slack_k:.2f} | Detections: {len(bandit.change_points)} | Total Reward: {sum(rewards):.1f}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Interactive widget
interact(
    run_cusum_bandit,
    threshold_h=FloatSlider(min=1.0, max=10.0, step=0.5, value=5.0, description='Threshold h:'),
    slack_k=FloatSlider(min=0.02, max=0.15, step=0.01, value=0.08, description='Slack k:')
);

## Summary

**What we learned:**

1. **CUSUM change-point detection** monitors cumulative deviations to detect regime shifts

2. **Combining CUSUM with bandits** enables immediate reset and re-exploration when changes detected

3. **Threshold tuning matters:**
   - Too sensitive (low h) → false alarms, over-resetting
   - Too insensitive (high h) → missed changes, slow adaptation
   - Sweet spot: h ≈ 4-6 for typical commodity applications

4. **Faster adaptation than discounting:** CUSUM can detect and restart within 10-20 steps, vs 50-100 for exponential forgetting

5. **Trade-off:** Computational overhead (monitoring CUSUM stats) vs faster recovery from regime shifts

**Commodity insight:** Abrupt market shocks (COVID, energy crisis, geopolitical events) are better handled by change-point detection than gradual discounting.

**Next steps:**
- Notebook 3: Apply to real commodity data with historical regime shifts
- Combine CUSUM with contextual bandits (Module 3)
- Explore Bayesian change-point detection for smoother detection

In [ ]:
key_takeaways(["CUSUM change-point detection", "Combining CUSUM with bandits", "Threshold tuning matters:", "Faster adaptation than discounting:", "Trade-off:"])